# ASR text embedding — Qwen3 offline Kaggle

Bản offline của notebook ASR embedding. Notebook không dùng Hugging Face Hub hoặc PyPI internet: model phải được attach từ Kaggle Models, còn các package phải có wheel trong Kaggle Dataset.

Mỗi ASR segment tạo một vector riêng. Hai model có dimension khác nhau nên output và run được tách theo model. Notebook vẫn giữ cấu trúc artifact, validation và latency benchmark của bản v1.

## 1. Offline package installation

Attach Dataset chứa các file `.whl`, chỉnh `WHEEL_ROOT` nếu auto-discovery không tìm đúng. Cell dùng `--no-index`, vì vậy pip sẽ không truy cập internet. Sau khi cài package, restart kernel/session rồi chạy notebook từ đầu.

In [1]:
import sys
import subprocess

# Đường dẫn tới Dataset offline vừa đính kèm
WHEEL_PATH = "/kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312-full"

# Cài đặt từ thư mục wheels không cần mạng và không sợ thiếu dependency
command = [
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", WHEEL_PATH,
    "sentence-transformers", "transformers", "accelerate"
]

subprocess.run(command, check=True)

Looking in links: /kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312-full


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-index', '--find-links', '/kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312-full', 'sentence-transformers', 'transformers', 'accelerate'], returncode=0)

In [2]:
# !pip install --no-cache-dir --no-deps /kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312/asr-embedding-wheels-py312/transformers-4.49.0-py3-none-any.whl
# !pip install --no-cache-dir --no-deps /kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312/asr-embedding-wheels-py312/sentence_transformers-3.4.1-py3-none-any.whl
# !pip install --no-cache-dir --no-deps /kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312/asr-embedding-wheels-py312/accelerate-1.4.0-py3-none-any.whl

In [3]:
# import os
# import subprocess
# import sys
# from pathlib import Path

# # Set this to the directory containing wheel files, or leave None for auto-discovery.
# WHEEL_ROOT = "/kaggle/input/datasets/annguyentranthien21/asr-embedding-wheels-py312/asr-embedding-wheels-py312"
# OFFLINE_REQUIREMENTS = [
#     'sentence-transformers>=3.0,<6',
#     'transformers>=4.51.0,<5',
#     'accelerate>=1.0.0,<2',
# ]

# # Make accidental Hub access fail fast for all later cells.
# os.environ['HF_HUB_OFFLINE'] = '1'
# os.environ['TRANSFORMERS_OFFLINE'] = '1'
# os.environ['HF_DATASETS_OFFLINE'] = '1'
# os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# def wheel_files(root: Path) -> list[Path]:
#     return sorted(path for path in root.rglob('*.whl') if path.is_file())

# def discover_wheel_files() -> list[Path]:
#     if WHEEL_ROOT:
#         root = Path(WHEEL_ROOT).expanduser()
#         if not root.is_dir():
#             raise FileNotFoundError(f'WHEEL_ROOT does not exist: {root}')
#         return wheel_files(root)
#     kaggle_root = Path('/kaggle/input')
#     if not kaggle_root.is_dir():
#         raise FileNotFoundError('Kaggle input root does not exist and WHEEL_ROOT is None')
#     return sorted(path for path in kaggle_root.rglob('*.whl') if path.is_file())

# def normalized_wheel_stem(path: Path) -> str:
#     return path.name.lower().replace('_', '-')

# wheel_paths = discover_wheel_files()
# if not wheel_paths:
#     raise FileNotFoundError('No .whl files found. Attach the offline wheel Dataset or set WHEEL_ROOT.')
# required_prefixes = {
#     'sentence-transformers': 'sentence-transformers-',
#     'transformers': 'transformers-',
#     'accelerate': 'accelerate-',
# }
# missing = [
#     package for package, prefix in required_prefixes.items()
#     if not any(normalized_wheel_stem(path).startswith(prefix) for path in wheel_paths)
# ]
# if missing:
#     raise FileNotFoundError(f'Missing direct-package wheels: {missing}. Found: {[path.name for path in wheel_paths]}')

# find_links = sorted({str(path.parent) for path in wheel_paths})
# command = [sys.executable, '-m', 'pip', 'install', '--no-index', '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed']
# for link in find_links:
#     command.extend(['--find-links', link])
# command.extend(OFFLINE_REQUIREMENTS)
# print('Offline wheel directories:', find_links)
# print('Running:', ' '.join(command))
# subprocess.run(command, check=True)
# print('Installation completed. Restart the Kaggle kernel/session before importing these packages.')

In [4]:
import numpy
import scipy
import sklearn
import sentence_transformers

print(numpy.__version__)
print(scipy.__version__)
print(sklearn.__version__)
print(sentence_transformers.__version__)

2.0.2
1.16.3
1.6.1
5.4.1


## 2. Offline configuration

`MODEL_PATHS` có thể điền path cụ thể của Kaggle Model. Nếu để `None`, notebook tìm local model dưới `/kaggle/input` theo tên Qwen3 4B/8B. Không dùng `model_name` để tải từ Hub.

Để chạy thành hai notebook/run riêng, chọn một model trong `MODELS_TO_RUN` và dùng `OUTPUT_ROOT_BASE` riêng.

In [5]:
from __future__ import annotations

import gc
import inspect
import json
import math
import os
import re
import subprocess
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch
from tqdm.auto import tqdm

MODEL_SPECS = {
    'qwen3-embedding-4b': {
        'model_name': 'Qwen/Qwen3-Embedding-4B',
        'embedding_dim': 2560,
        'batch_size': 8,
        'env_path': 'ASR_QWEN3_4B_MODEL_PATH',
        'hints': ('qwen3', 'embedding', '4b'),
    },
    'qwen3-embedding-8b': {
        'model_name': 'Qwen/Qwen3-Embedding-8B',
        'embedding_dim': 4096,
        'batch_size': 2,
        'env_path': 'ASR_QWEN3_8B_MODEL_PATH',
        'hints': ('qwen3', 'embedding', '8b'),
    },
}

# Run one or both sequentially. For two independent runs, choose one key each time.
MODELS_TO_RUN = ['qwen3-embedding-4b', 'qwen3-embedding-8b']
MODEL_PATHS = {
    'qwen3-embedding-4b': "/kaggle/input/models/annguyentranthien21/qwen3-embedding-4b/transformers/default/1",
    'qwen3-embedding-8b': "/kaggle/input/models/annguyentranthien21/qwen3-embedding-8b/transformers/default/1",
}
INPUT_ROOT = "/kaggle/input/datasets/nguyentranthienan/aic-2026-asr-corrected/asr_output_corrected_split"
OUTPUT_ROOT_BASE = None
MAX_SEQUENCE_LENGTH = 2048
ENCODE_CHUNK_SIZE = 4096
NORMALIZE_EMBEDDINGS = True
WARMUP_SEGMENTS = 32
USE_FLASH_ATTENTION_2 = False
DEVICE = "cuda"
USE_DEVICE_MAP_AUTO = False
torch.cuda.set_device(0)
ALLOW_NONEMPTY_OUTPUT = False

# Optional explicit paths for the input/output.
if os.environ.get('ASR_INPUT_ROOT', '').strip():
    INPUT_ROOT = os.environ['ASR_INPUT_ROOT']
if os.environ.get('ASR_OUTPUT_ROOT', '').strip():
    OUTPUT_ROOT_BASE = os.environ['ASR_OUTPUT_ROOT']
DEVICE = os.environ.get('ASR_EMBEDDING_DEVICE', '').strip() or DEVICE

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def model_dir_has_weights(path: Path) -> bool:
    if not (path / 'config.json').is_file():
        return False
    names = {item.name for item in path.iterdir()}
    return bool(
        'modules.json' in names
        or 'model.safetensors' in names
        or 'model.safetensors.index.json' in names
        or 'pytorch_model.bin' in names
        or 'pytorch_model.bin.index.json' in names
        or any(name.endswith('.safetensors') for name in names)
    )

def resolve_local_model_path(model_key: str) -> Path:
    spec = MODEL_SPECS[model_key]
    configured = MODEL_PATHS.get(model_key) or os.environ.get(spec['env_path'], '').strip()
    if configured:
        candidate = Path(configured).expanduser()
        if not candidate.is_dir():
            raise FileNotFoundError(f'{model_key} model path does not exist: {candidate}')
        if model_dir_has_weights(candidate):
            return candidate
        nested = sorted(path for path in candidate.rglob('*') if path.is_dir() and model_dir_has_weights(path))
        if len(nested) == 1:
            return nested[0]
        raise FileNotFoundError(f'No unambiguous local model snapshot under {candidate}: {nested[:10]}')

    root = Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError('Kaggle input root does not exist; set MODEL_PATHS explicitly.')
    candidates = []
    for config in root.rglob('config.json'):
        path = config.parent
        label = path.as_posix().lower().replace('_', '-')
        if model_dir_has_weights(path) and all(hint in label for hint in spec['hints']):
            candidates.append(path)
    candidates = sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(f'Could not resolve exactly one local path for {model_key}. Attach the Kaggle Model or set MODEL_PATHS[{model_key!r}]. Candidates: {candidates[:20]}')

def _default_input_root() -> Path:
    if INPUT_ROOT:
        return Path(INPUT_ROOT).expanduser()
    local = Path.cwd() / 'asr_output_corrected'
    if local.is_dir() and any(local.rglob('asr_segments.jsonl')):
        return local
    manifests = sorted(Path('/kaggle/input').rglob('asr_segments.jsonl'))
    if manifests:
        return manifests[0].parent.parent
    return Path('/kaggle/input/asr-output-corrected')

def _default_output_root() -> Path:
    if OUTPUT_ROOT_BASE:
        return Path(OUTPUT_ROOT_BASE).expanduser()
    if Path('/kaggle/working').is_dir():
        return Path('/kaggle/working/asr_embedding_output_qwen3_offline_v1')
    return Path.cwd() / 'asr_embedding_output_qwen3_offline_v1'

INPUT_ROOT = _default_input_root()
OUTPUT_ROOT_BASE = _default_output_root()
LOCAL_MODEL_PATHS = {key: resolve_local_model_path(key) for key in MODELS_TO_RUN}
for key in MODELS_TO_RUN:
    if key not in MODEL_SPECS:
        raise KeyError(f'Unknown model key: {key}')
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable. Check the Kaggle accelerator.')
if USE_DEVICE_MAP_AUTO:
    raise ValueError('Offline notebook is configured for one GPU; set USE_DEVICE_MAP_AUTO=False.')

print('INPUT_ROOT:', INPUT_ROOT)
print('OUTPUT_ROOT_BASE:', OUTPUT_ROOT_BASE)
print('MODELS_TO_RUN:', MODELS_TO_RUN)
print('DEVICE:', DEVICE)
print('LOCAL_MODEL_PATHS:', {key: str(path) for key, path in LOCAL_MODEL_PATHS.items()})
print('GPU count:', torch.cuda.device_count())
print('Offline flags:', {key: os.environ.get(key) for key in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE')})

INPUT_ROOT: /kaggle/input/datasets/nguyentranthienan/aic-2026-asr-corrected/asr_output_corrected_split
OUTPUT_ROOT_BASE: /kaggle/working/asr_embedding_output_qwen3_offline_v1
MODELS_TO_RUN: ['qwen3-embedding-4b', 'qwen3-embedding-8b']
DEVICE: cuda
LOCAL_MODEL_PATHS: {'qwen3-embedding-4b': '/kaggle/input/models/annguyentranthien21/qwen3-embedding-4b/transformers/default/1', 'qwen3-embedding-8b': '/kaggle/input/models/annguyentranthien21/qwen3-embedding-8b/transformers/default/1'}
GPU count: 1
Offline flags: {'HF_HUB_OFFLINE': None, 'TRANSFORMERS_OFFLINE': None, 'HF_DATASETS_OFFLINE': None}


## 3. Load and flatten ASR segments

Đoạn này giữ nguyên logic v1: một segment một vector, giữ text gốc để audit, cleanup lặp liên tiếp nhẹ và sắp xếp segment ổn định theo timestamp.

In [6]:
_TOKEN_RE = re.compile(r'\S+')
_SAFE_ID_RE = re.compile(r'^[A-Za-z0-9][A-Za-z0-9._-]*$')

def optional_float(value: Any) -> float | None:
    if value is None or value == '':
        return None
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    return result if math.isfinite(result) else None

def required_float(value: Any, name: str) -> float:
    result = optional_float(value)
    if result is None:
        raise ValueError(f'{name} must be a finite number')
    return result

def clean_embedding_text(value: Any) -> dict[str, Any]:
    text = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(value or ''))).strip()
    tokens = _TOKEN_RE.findall(text)
    cleaned: list[str] = []
    repeat_detected = False
    repeat_runs = 0
    removed = 0
    index = 0
    while index < len(tokens):
        found = None
        max_width = min(8, (len(tokens) - index) // 3)
        for width in range(1, max_width + 1):
            phrase = [unicodedata.normalize('NFKC', token).casefold() for token in tokens[index:index + width]]
            repeats = 1
            while index + (repeats + 1) * width <= len(tokens):
                other = [unicodedata.normalize('NFKC', token).casefold() for token in tokens[index + repeats * width:index + (repeats + 1) * width]]
                if other != phrase:
                    break
                repeats += 1
            if repeats >= 3 and (found is None or width * repeats > found[0]):
                found = (width * repeats, width)
        if found is None:
            cleaned.append(tokens[index])
            index += 1
        else:
            covered, width = found
            cleaned.extend(tokens[index:index + width])
            removed += covered - width
            repeat_runs += 1
            repeat_detected = True
            index += covered
    return {
        'embedding_text': ' '.join(cleaned).strip(),
        'raw_word_count': len(tokens),
        'clean_word_count': len(cleaned),
        'repeat_detected': repeat_detected,
        'repeat_runs': repeat_runs,
        'repeat_tokens_removed': removed,
    }

def artifact_stem(video_id: str) -> str:
    if not _SAFE_ID_RE.fullmatch(video_id):
        raise ValueError(f'Unsafe video_id for filename: {video_id!r}')
    return video_id

def load_asr_segments(root: Path) -> tuple[dict[str, dict[str, Any]], dict[str, Any]]:
    manifests = sorted(root.rglob('asr_segments.jsonl'))
    if not manifests:
        raise FileNotFoundError(f'No asr_segments.jsonl below {root}')
    videos: dict[str, dict[str, Any]] = {}
    seen_ids: set[str] = set()
    errors: list[str] = []
    for manifest in manifests:
        source_file = str(manifest.relative_to(root))
        with manifest.open('r', encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    video = json.loads(line)
                except json.JSONDecodeError as exc:
                    errors.append(f'{source_file}:{line_number}: {exc}')
                    continue
                if not isinstance(video, dict):
                    errors.append(f'{source_file}:{line_number}: record is not an object')
                    continue
                video_id = str(video.get('video_id') or '').strip()
                if not video_id or video_id in videos:
                    errors.append(f'{source_file}:{line_number}: invalid or duplicate video_id {video_id!r}')
                    continue
                info = {
                    'video_id': video_id,
                    'batch_id': str(video.get('batch_id') or manifest.parent.name),
                    'dataset_code': video.get('dataset_code'),
                    'source_file': source_file,
                    'video_duration_seconds': optional_float((video.get('source') or {}).get('duration_seconds')),
                    'segments': [],
                }
                videos[video_id] = info
                segments = video.get('segments') or []
                if not isinstance(segments, list):
                    errors.append(f'{source_file}:{line_number}: segments is not a list')
                    continue
                for segment_index, segment in enumerate(segments):
                    location = f'{source_file}:{line_number}:segment[{segment_index}]'
                    if not isinstance(segment, dict):
                        errors.append(f'{location}: not an object')
                        continue
                    segment_id = str(segment.get('segment_id') or '').strip()
                    text = str(segment.get('text') or '').strip()
                    if not segment_id or segment_id in seen_ids or not text:
                        errors.append(f'{location}: missing, duplicate segment_id, or empty text')
                        continue
                    seen_ids.add(segment_id)
                    try:
                        start = required_float(segment.get('start_seconds'), 'start_seconds')
                        end = required_float(segment.get('end_seconds'), 'end_seconds')
                    except ValueError as exc:
                        errors.append(f'{location}: {exc}')
                        continue
                    if start < 0 or end < start:
                        errors.append(f'{location}: invalid range {start}..{end}')
                        continue
                    cleaned = clean_embedding_text(text)
                    if not cleaned['embedding_text']:
                        errors.append(f'{location}: cleanup produced empty text')
                        continue
                    info['segments'].append({
                        'embedding_index_0': -1, 'segment_order_0': -1,
                        'segment_id': segment_id, 'video_id': video_id,
                        'batch_id': info['batch_id'], 'dataset_code': info['dataset_code'],
                        'start': start, 'end': end, 'start_seconds': start, 'end_seconds': end,
                        'text': text, 'embedding_text': cleaned['embedding_text'],
                        'normalized_text': str(segment.get('normalized_text') or ''),
                        'raw_text': str(segment.get('raw_text') or ''),
                        'status': str(segment.get('correction_status') or 'unknown'),
                        'correction_status': str(segment.get('correction_status') or 'unknown'),
                        'confidence': optional_float(segment.get('confidence')),
                        'source_file': source_file, 'source_manifest_line': line_number,
                        'source_segment_index': segment_index,
                        'video_duration_seconds': info['video_duration_seconds'], **cleaned,
                    })
    if errors:
        raise RuntimeError(f'Input validation failed with {len(errors)} errors.\n' + '\n'.join(errors[:30]))
    for info in videos.values():
        info['segments'].sort(key=lambda row: (row['start_seconds'], row['end_seconds'], row['segment_id']))
        for order, row in enumerate(info['segments']):
            row['embedding_index_0'] = order
            row['segment_order_0'] = order
    all_segments = [row for info in videos.values() for row in info['segments']]
    summary = {
        'manifest_count': len(manifests), 'video_count': len(videos),
        'segment_count': len(all_segments),
        'empty_video_count': sum(not info['segments'] for info in videos.values()),
        'empty_text_segment_count': 0,
        'repeat_detected_segment_count': sum(row['repeat_detected'] for row in all_segments),
        'total_raw_word_count': sum(row['raw_word_count'] for row in all_segments),
        'total_clean_word_count': sum(row['clean_word_count'] for row in all_segments),
        'total_repeat_tokens_removed': sum(row['repeat_tokens_removed'] for row in all_segments),
        'status_counts': dict(Counter(row['status'] for row in all_segments)),
    }
    return dict(sorted(videos.items())), summary

VIDEOS_BY_ID, LOAD_SUMMARY = load_asr_segments(INPUT_ROOT)
print(json.dumps(LOAD_SUMMARY, ensure_ascii=False, indent=2))

{
  "manifest_count": 10,
  "video_count": 873,
  "segment_count": 41201,
  "empty_video_count": 20,
  "empty_text_segment_count": 0,
  "repeat_detected_segment_count": 134,
  "total_raw_word_count": 1456467,
  "total_clean_word_count": 1450194,
  "total_repeat_tokens_removed": 6273,
  "status_counts": {
    "corrected": 41176,
    "raw_fallback": 25
  }
}


## 4. Load local Qwen model

Model được load bằng filesystem path với `local_files_only=True`. `device_map=auto` bị tắt để tránh lỗi model ở `cuda:1` nhưng input ở `cuda:0`; trên RTX 6000 48 GB, mỗi model chạy trọn trên `cuda:0`.

In [7]:
import sentence_transformers
import transformers
from sentence_transformers import SentenceTransformer

def synchronize_cuda() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize(torch.device(DEVICE))

def reset_peak_memory_stats() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(torch.device(DEVICE))

def peak_gpu_memory_gib() -> float | None:
    if not torch.cuda.is_available():
        return None
    synchronize_cuda()
    return float(torch.cuda.max_memory_allocated(torch.device(DEVICE)) / (1024 ** 3))

def model_output_root(model_key: str) -> Path:
    return OUTPUT_ROOT_BASE / model_key

def load_local_qwen_model(model_key: str) -> tuple[SentenceTransformer, dict[str, Any]]:
    spec = MODEL_SPECS[model_key]
    model_path = LOCAL_MODEL_PATHS[model_key]
    dtype = torch.float16 if DEVICE.startswith('cuda') else torch.float32
    transformers_major = int(transformers.__version__.split('.')[0])
    dtype_key = 'dtype' if transformers_major >= 5 else 'torch_dtype'
    model_kwargs: dict[str, Any] = {dtype_key: dtype}
    if USE_FLASH_ATTENTION_2:
        model_kwargs['attn_implementation'] = 'flash_attention_2'
    constructor_kwargs: dict[str, Any] = {
        'device': DEVICE, 'local_files_only': True, 'trust_remote_code': False,
        'model_kwargs': model_kwargs,
    }
    processor_payload = {'padding_side': 'left', 'local_files_only': True}
    signature = inspect.signature(SentenceTransformer.__init__).parameters
    if 'processor_kwargs' in signature:
        constructor_kwargs['processor_kwargs'] = processor_payload
    else:
        constructor_kwargs['tokenizer_kwargs'] = processor_payload
    model = SentenceTransformer(str(model_path), **constructor_kwargs)
    if hasattr(model, 'max_seq_length'):
        model.max_seq_length = MAX_SEQUENCE_LENGTH
    model.eval()
    actual_dim = model.get_embedding_dimension() if hasattr(model, 'get_embedding_dimension') else model.get_sentence_embedding_dimension()
    if actual_dim != spec['embedding_dim']:
        expected_dim = spec['embedding_dim']
        raise ValueError(f'{model_key} returned dim {actual_dim}; expected {expected_dim}')
    return model, {
        'actual_dim': actual_dim, 'dtype': str(dtype).replace('torch.', ''),
        'model_path': str(model_path), 'local_files_only': True,
        'max_sequence_length': getattr(model, 'max_seq_length', MAX_SEQUENCE_LENGTH),
    }

print('sentence-transformers:', sentence_transformers.__version__)
print('transformers:', transformers.__version__)
print('MODELS:', {key: MODEL_SPECS[key] for key in MODELS_TO_RUN})

sentence-transformers: 5.4.1
transformers: 5.0.0
MODELS: {'qwen3-embedding-4b': {'model_name': 'Qwen/Qwen3-Embedding-4B', 'embedding_dim': 2560, 'batch_size': 8, 'env_path': 'ASR_QWEN3_4B_MODEL_PATH', 'hints': ('qwen3', 'embedding', '4b')}, 'qwen3-embedding-8b': {'model_name': 'Qwen/Qwen3-Embedding-8B', 'embedding_dim': 4096, 'batch_size': 2, 'env_path': 'ASR_QWEN3_8B_MODEL_PATH', 'hints': ('qwen3', 'embedding', '8b')}}


## 5. Encode, write and measure latency

Mỗi model có thư mục riêng. Latency là synchronized encode wall-clock, không tính model load, warmup hoặc thời gian ghi artifact.

In [8]:
def atomic_write_json(path: Path, payload: Any) -> None:
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
    temporary.replace(path)

def atomic_write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')
    temporary.replace(path)

def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        np.save(handle, array, allow_pickle=False)
    temporary.replace(path)

def benchmark_and_write(model_key: str) -> dict[str, Any]:
    spec = MODEL_SPECS[model_key]
    output_root = model_output_root(model_key)
    if output_root.exists() and any(output_root.iterdir()) and not ALLOW_NONEMPTY_OUTPUT:
        raise FileExistsError(f'Output directory is not empty: {output_root}. Choose a new output path.')
    (output_root / 'embeddings').mkdir(parents=True, exist_ok=True)
    (output_root / 'map-segments').mkdir(parents=True, exist_ok=True)
    video_ids = sorted(VIDEOS_BY_ID)
    ordered_segments = [row for video_id in video_ids for row in VIDEOS_BY_ID[video_id]['segments']]
    texts = [row['embedding_text'] for row in ordered_segments]
    model = None
    try:
        reset_peak_memory_stats()
        started = time.perf_counter()
        model, model_meta = load_local_qwen_model(model_key)
        synchronize_cuda()
        load_seconds = time.perf_counter() - started
        warmup_count = min(WARMUP_SEGMENTS, len(texts))
        warmup_started = time.perf_counter()
        if warmup_count:
            model.encode(texts[:warmup_count], batch_size=spec['batch_size'], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=NORMALIZE_EMBEDDINGS)
            synchronize_cuda()
        warmup_seconds = time.perf_counter() - warmup_started
        vectors: list[np.ndarray] = []
        chunk_latencies: list[float] = []
        encode_started = time.perf_counter()
        for offset in tqdm(range(0, len(texts), ENCODE_CHUNK_SIZE), desc=f'Encoding {model_key}'):
            chunk_texts = texts[offset:offset + ENCODE_CHUNK_SIZE]
            synchronize_cuda()
            chunk_started = time.perf_counter()
            chunk = model.encode(chunk_texts, batch_size=spec['batch_size'], show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=NORMALIZE_EMBEDDINGS)
            synchronize_cuda()
            chunk_latencies.append((time.perf_counter() - chunk_started) * 1000.0)
            chunk = np.asarray(chunk, dtype=np.float32)
            if chunk.ndim == 1:
                chunk = chunk.reshape(1, -1)
            expected = (len(chunk_texts), spec['embedding_dim'])
            if chunk.shape != expected or not np.isfinite(chunk).all():
                raise ValueError(f'Invalid {model_key} chunk at {offset}: shape={chunk.shape}, expected={expected}')
            vectors.append(chunk)
        encode_seconds = time.perf_counter() - encode_started
        all_vectors = np.concatenate(vectors, axis=0).astype(np.float32, copy=False) if vectors else np.empty((0, spec['embedding_dim']), dtype=np.float32)
        if all_vectors.shape != (len(ordered_segments), spec['embedding_dim']):
            raise ValueError(f'Global vector shape mismatch: {all_vectors.shape}')
        cursor = 0
        for video_id in tqdm(video_ids, desc=f'Writing {model_key}'):
            segments = VIDEOS_BY_ID[video_id]['segments']
            count = len(segments)
            stem = artifact_stem(video_id)
            atomic_save_npy(output_root / 'embeddings' / f'{stem}.npy', all_vectors[cursor:cursor + count])
            atomic_write_jsonl(output_root / 'map-segments' / f'{stem}.jsonl', segments)
            cursor += count
        return {
            'model_key': model_key, 'model_name': spec['model_name'], 'model_path': model_meta['model_path'],
            'embedding_dim': spec['embedding_dim'], 'device': DEVICE, 'dtype': model_meta['dtype'],
            'local_files_only': True, 'max_sequence_length': model_meta['max_sequence_length'],
            'batch_size': spec['batch_size'], 'encode_chunk_size': ENCODE_CHUNK_SIZE,
            'num_videos': len(video_ids), 'num_segments': len(ordered_segments),
            'model_load_seconds': float(load_seconds), 'warmup_seconds': float(warmup_seconds),
            'warmup_segments': warmup_count, 'encode_seconds': float(encode_seconds),
            'latency_ms_per_segment': float(encode_seconds * 1000.0 / len(ordered_segments)) if ordered_segments else 0.0,
            'throughput_segments_per_second': float(len(ordered_segments) / encode_seconds) if encode_seconds > 0 else 0.0,
            'chunk_count': len(chunk_latencies),
            'chunk_latency_ms_p50': float(np.percentile(chunk_latencies, 50)) if chunk_latencies else 0.0,
            'chunk_latency_ms_p95': float(np.percentile(chunk_latencies, 95)) if chunk_latencies else 0.0,
            'chunk_latency_ms_p99': float(np.percentile(chunk_latencies, 99)) if chunk_latencies else 0.0,
            'chunk_latency_ms': chunk_latencies,
            'latency_definition': 'synchronized wall-clock encode time, excluding model load and warmup',
            'peak_gpu_memory_gib': peak_gpu_memory_gib(), 'output_root': str(output_root),
        }
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

BENCHMARK_RESULTS = {}
for model_key in MODELS_TO_RUN:
    print('\n' + '=' * 80)
    print('RUNNING:', model_key, MODEL_SPECS[model_key]['model_name'])
    BENCHMARK_RESULTS[model_key] = benchmark_and_write(model_key)
    print(json.dumps(BENCHMARK_RESULTS[model_key], ensure_ascii=False, indent=2))


RUNNING: qwen3-embedding-4b Qwen/Qwen3-Embedding-4B


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding qwen3-embedding-4b:   0%|          | 0/11 [00:00<?, ?it/s]

Writing qwen3-embedding-4b:   0%|          | 0/873 [00:00<?, ?it/s]

{
  "model_key": "qwen3-embedding-4b",
  "model_name": "Qwen/Qwen3-Embedding-4B",
  "model_path": "/kaggle/input/models/annguyentranthien21/qwen3-embedding-4b/transformers/default/1",
  "embedding_dim": 2560,
  "device": "cuda",
  "dtype": "float16",
  "local_files_only": true,
  "max_sequence_length": 2048,
  "batch_size": 8,
  "encode_chunk_size": 4096,
  "num_videos": 873,
  "num_segments": 41201,
  "model_load_seconds": 10.895831300999987,
  "warmup_seconds": 0.9206947369999625,
  "warmup_segments": 32,
  "encode_seconds": 126.61654345100004,
  "latency_ms_per_segment": 3.0731424832164276,
  "throughput_segments_per_second": 325.39981646193473,
  "chunk_count": 11,
  "chunk_latency_ms_p50": 12162.478415999998,
  "chunk_latency_ms_p95": 14946.69274649999,
  "chunk_latency_ms_p99": 14964.819709300025,
  "chunk_latency_ms": [
    14969.351450000033,
    12162.478415999998,
    10654.83342600004,
    10910.204482000041,
    10265.944721999971,
    11916.714626999976,
    13906.16151199

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Encoding qwen3-embedding-8b:   0%|          | 0/11 [00:00<?, ?it/s]

Writing qwen3-embedding-8b:   0%|          | 0/873 [00:00<?, ?it/s]

{
  "model_key": "qwen3-embedding-8b",
  "model_name": "Qwen/Qwen3-Embedding-8B",
  "model_path": "/kaggle/input/models/annguyentranthien21/qwen3-embedding-8b/transformers/default/1",
  "embedding_dim": 4096,
  "device": "cuda",
  "dtype": "float16",
  "local_files_only": true,
  "max_sequence_length": 2048,
  "batch_size": 2,
  "encode_chunk_size": 4096,
  "num_videos": 873,
  "num_segments": 41201,
  "model_load_seconds": 19.225364665999905,
  "warmup_seconds": 0.3417015560000891,
  "warmup_segments": 32,
  "encode_seconds": 422.1879916529999,
  "latency_ms_per_segment": 10.247032636416591,
  "throughput_segments_per_second": 97.589227582445,
  "chunk_count": 11,
  "chunk_latency_ms_p50": 41470.84112499999,
  "chunk_latency_ms_p95": 44069.07090449994,
  "chunk_latency_ms_p99": 44316.618453699935,
  "chunk_latency_ms": [
    43759.63646799994,
    41470.84112499999,
    40113.71167200002,
    40348.57369500003,
    39789.99427600001,
    41366.042588000026,
    43244.70609299999,
    

## 6. Validate artifacts and write metadata

In [9]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                row = json.loads(line)
                if not isinstance(row, dict):
                    raise AssertionError(f'Mapping row is not object: {path}:{line_number}')
                rows.append(row)
    return rows

def validate_outputs(model_key: str) -> dict[str, Any]:
    spec = MODEL_SPECS[model_key]
    root = model_output_root(model_key)
    embedding_dir = root / 'embeddings'
    mapping_dir = root / 'map-segments'
    expected = {artifact_stem(video_id) for video_id in VIDEOS_BY_ID}
    assert {path.stem for path in embedding_dir.glob('*.npy')} == expected
    assert {path.stem for path in mapping_dir.glob('*.jsonl')} == expected
    vectors = 0
    rows = 0
    max_norm_error = 0.0
    for video_id in sorted(VIDEOS_BY_ID):
        matrix = np.load(embedding_dir / f'{video_id}.npy', mmap_mode='r', allow_pickle=False)
        mapping = read_jsonl(mapping_dir / f'{video_id}.jsonl')
        source_segments = VIDEOS_BY_ID[video_id]['segments']
        assert matrix.dtype == np.float32 and matrix.ndim == 2
        assert matrix.shape == (len(source_segments), spec['embedding_dim'])
        assert len(mapping) == matrix.shape[0] and np.isfinite(matrix).all()
        assert [row['segment_id'] for row in mapping] == [row['segment_id'] for row in source_segments]
        assert len({row['segment_id'] for row in mapping}) == len(mapping)
        for index, row in enumerate(mapping):
            assert row['embedding_index_0'] == index and row['segment_order_0'] == index
            assert row['video_id'] == video_id and row['text'] and row['embedding_text'] and row['source_file']
        if len(matrix) and NORMALIZE_EMBEDDINGS:
            norms = np.linalg.norm(np.asarray(matrix), axis=1)
            error = float(np.max(np.abs(norms - 1.0)))
            max_norm_error = max(max_norm_error, error)
            assert np.all(norms > 1e-8) and error <= 1e-3
        vectors += matrix.shape[0]
        rows += len(mapping)
    assert vectors == LOAD_SUMMARY['segment_count'] and rows == vectors
    assert not list(root.rglob('*.tmp'))
    return {'model_key': model_key, 'video_count': len(VIDEOS_BY_ID), 'vector_count': vectors, 'mapping_row_count': rows, 'embedding_dim': spec['embedding_dim'], 'max_l2_norm_error': max_norm_error}

VALIDATION_REPORTS = {key: validate_outputs(key) for key in MODELS_TO_RUN}
created_at = utc_now_iso()
run_id = 'asr_text_embedding_qwen3_offline_v1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
all_segments = [row for info in VIDEOS_BY_ID.values() for row in info['segments']]
status_counts = dict(Counter(row['status'] for row in all_segments))
for model_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    root = model_output_root(model_key)
    latency = BENCHMARK_RESULTS[model_key]
    validation = VALIDATION_REPORTS[model_key]
    model_info = {
        'artifact_schema_version': 'aic.asr_text_embedding.offline.v1', 'run_id': run_id, 'created_at': created_at,
        'model_key': model_key, 'model_name': spec['model_name'], 'model_path': str(LOCAL_MODEL_PATHS[model_key]),
        'local_files_only': True, 'sentence_transformers_version': sentence_transformers.__version__,
        'transformers_version': transformers.__version__, 'embedding_dim': spec['embedding_dim'],
        'dtype': latency['dtype'], 'normalize_embeddings': NORMALIZE_EMBEDDINGS, 'similarity_metric': 'cosine',
        'max_sequence_length': MAX_SEQUENCE_LENGTH, 'source_text_field': 'text', 'encoded_text_field': 'embedding_text',
        'output_layout': {'embeddings': 'embeddings/<video_id>.npy', 'mapping': 'map-segments/<video_id>.jsonl', 'row_key': 'embedding_index_0'},
        'latency': latency, 'validation': validation,
    }
    summary = {**LOAD_SUMMARY, **validation, 'status': 'success', 'run_id': run_id, 'created_at': created_at, 'model_key': model_key, 'model_name': spec['model_name'], 'input_root': str(INPUT_ROOT), 'output_root': str(root), 'latency': latency, 'status_counts': status_counts}
    atomic_write_json(root / 'model_info.json', model_info)
    atomic_write_json(root / 'summary.json', summary)
    atomic_write_json(root / '_SUCCESS', {'status': 'success', 'run_id': run_id, 'model_key': model_key, 'vector_count': validation['vector_count'], 'embedding_dim': spec['embedding_dim']})
benchmark_summary = {'status': 'success', 'run_id': run_id, 'created_at': created_at, 'input_root': str(INPUT_ROOT), 'output_root': str(OUTPUT_ROOT_BASE), 'models': BENCHMARK_RESULTS, 'validation': VALIDATION_REPORTS, 'offline': True}
atomic_write_json(OUTPUT_ROOT_BASE / 'latency_benchmark.json', benchmark_summary)
atomic_write_json(OUTPUT_ROOT_BASE / '_SUCCESS', {'status': 'success', 'run_id': run_id, 'models': MODELS_TO_RUN, 'offline': True})
print('SUCCESS:', OUTPUT_ROOT_BASE)
print(json.dumps(benchmark_summary, ensure_ascii=False, indent=2))

SUCCESS: /kaggle/working/asr_embedding_output_qwen3_offline_v1
{
  "status": "success",
  "run_id": "asr_text_embedding_qwen3_offline_v1_20260903T161239Z",
  "created_at": "2026-09-03T16:12:39.706488Z",
  "input_root": "/kaggle/input/datasets/nguyentranthienan/aic-2026-asr-corrected/asr_output_corrected_split",
  "output_root": "/kaggle/working/asr_embedding_output_qwen3_offline_v1",
  "models": {
    "qwen3-embedding-4b": {
      "model_key": "qwen3-embedding-4b",
      "model_name": "Qwen/Qwen3-Embedding-4B",
      "model_path": "/kaggle/input/models/annguyentranthien21/qwen3-embedding-4b/transformers/default/1",
      "embedding_dim": 2560,
      "device": "cuda",
      "dtype": "float16",
      "local_files_only": true,
      "max_sequence_length": 2048,
      "batch_size": 8,
      "encode_chunk_size": 4096,
      "num_videos": 873,
      "num_segments": 41201,
      "model_load_seconds": 10.895831300999987,
      "warmup_seconds": 0.9206947369999625,
      "warmup_segments": 32,


## Kaggle checklist

1. Attach hai Kaggle Model local chứa Qwen3 Embedding 4B/8B, hoặc set `MODEL_PATHS` cụ thể.
2. Attach Dataset chứa wheel của `sentence-transformers`, `transformers`, `accelerate` và dependency wheels còn thiếu.
3. Chạy cell offline install, restart session/kernel, rồi chạy từ đầu.
4. Kiểm tra `LOCAL_MODEL_PATHS` và `DEVICE` trước khi encode.
5. Không dùng `pip install` không có `--no-index`, không truyền model ID Hub vào `SentenceTransformer`.